In [0]:
spark.conf.set(
  "fs.azure.account.key.salesviewacc.dfs.core.windows.net",
  "<accesskey>"
)

In [0]:
df = spark.read.csv(
  "abfss://bronze@salesviewacc.dfs.core.windows.net/sales_view/sales",
  header=True,
  inferSchema=True
)

In [0]:
import re

def to_snake_case(col):
    col = col.strip()
    col = re.sub(r'[^\w]+', '_', col)
    col = re.sub(r'(?<!^)(?=[A-Z])', '_', col).lower()
    return col

In [0]:
df = df.toDF(*[to_snake_case(c) for c in df.columns])


In [0]:
from pyspark.sql.functions import to_date,col
df = df.withColumn("order_date", to_date(col("order_date"))) \
       .withColumn("ship_date", to_date(col("ship_date")))


In [0]:
print(df.columns)

# write to silver


In [0]:
df.write.format("delta") \
  .mode("overwrite") \
  .save("abfss://silver@salesviewacc.dfs.core.windows.net/sales_view/customer_sales")
